In [ ]:
dataset_dir = "cars_after_2002"

In [ ]:
import os
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
import os
from torch.utils.data import Dataset
from PIL import Image
import glob

class CarModelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.labels = []
        self.transform = transform
        self.class_to_idx = {}
        idx = 0

        for class_name in sorted(os.listdir(root_dir)):
            class_path = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            if class_name not in self.class_to_idx:
                self.class_to_idx[class_name] = idx
                idx += 1
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
                for img_path in glob.glob(os.path.join(class_path, ext)):
                    self.samples.append(img_path)
                    self.labels.append(self.class_to_idx[class_name])

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        print(f"✅ Loaded {len(self.samples)} images from {len(self.class_to_idx)} classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB")  # Ensure it's a PIL image!
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
from torchvision import transforms


class RepeatGrayChannels:
    def __call__(self, tensor):
        return tensor.repeat(3, 1, 1)  # [1, H, W] → [3, H, W] from rgb to gray scale

# Transforms

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224,224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

transform_val = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


full_dataset = CarModelDataset(dataset_dir)

train_size = int(0.98 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

train_data.dataset.transform = transform_train
val_data.dataset.transform = transform_val

✅ Loaded 186289 images from 4481 classes.


In [ ]:
import os

def get_labels_from_folder(dataset_path):
    labels = []
    for item in os.listdir(dataset_path):
        full_path = os.path.join(dataset_path, item)
        if os.path.isdir(full_path):
            labels.append(item)
    return sorted(labels)

target_labels= get_labels_from_folder("flattened_cars_folder")
print(len(target_labels))

398


In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=4)


class_names = list(full_dataset.class_to_idx.keys())
num_classes = len(class_names)
print("Total class names:", num_classes)
images, labels = next(iter(train_loader))
print(images.shape)

Total class names: 4481
torch.Size([64, 1, 224, 224])


In [ ]:
print("🔍 Input shape:", images.shape)
print("🔍 Label shape:", labels.shape)

🔍 Input shape: torch.Size([64, 1, 224, 224])
🔍 Label shape: torch.Size([64])


In [ ]:
print("Max label:", max(full_dataset.labels))
print("Min label:", min(full_dataset.labels))
print("Expected range: 0 to", num_classes - 1)

Max label: 4480
Min label: 0
Expected range: 0 to 4480


In [ ]:
import torch
import torch.nn as nn
from timm import create_model

model_name = 'efficientnetv2_m'



model = create_model(model_name, pretrained=False, num_classes=num_classes)



old_conv = model.conv_stem

model.conv_stem = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None
)

state_dict = torch.load("saved_models/fold5_epoch2.pth", map_location=device)
model.load_state_dict(state_dict)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)



/leonardo/pub/userexternal/sdigioia/sdigioia/env/Gabenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
print("Ready For Train")

Ready For Train


In [ ]:
from collections import Counter
import torch

label_counts = Counter(full_dataset.labels)
total_samples = len(full_dataset)
num_classes = len(full_dataset.class_to_idx)

In [ ]:
class_weights = []
for i in range(num_classes):
    class_count = label_counts.get(i, 0)
    if class_count == 0:
        class_weights.append(0.0)
    else:
        weight = total_samples / (num_classes * class_count)
        class_weights.append(weight)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import os

base_dir = "saved_models"
os.makedirs(base_dir, exist_ok=True)
k = 5
num_epochs = 2
batch_size = 64
save_every_n_epochs = 1

all_preds, all_labels = [], []


kf = KFold(n_splits=k, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset)):
    print(f"\nFold {fold+1}/{k} ------------------------")

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    train_subset.dataset.transform = transform_train
    val_subset.dataset.transform = transform_val

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=4)

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"[Fold {fold+1}] Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_train_acc = correct / total
        print(f"Epoch {epoch+1} | Train Acc: {epoch_train_acc:.4f}")

        if (epoch + 1) % save_every_n_epochs == 0:
            save_path = os.path.join(base_dir, f"fold{fold+1}_epoch{epoch+1}.pth")
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to: {save_path}")

        model.eval()
        val_correct, val_total = 0, 0
        fold_preds, fold_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)

                fold_preds.extend(preds.cpu().tolist())
                fold_labels.extend(labels.cpu().tolist())

                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        acc = val_correct / val_total
        print(f"Validation Acc (Fold {fold+1}, Epoch {epoch+1}): {acc:.4f}")

        all_preds.extend(fold_preds)
        all_labels.extend(fold_labels)

print("\nFinal Evaluation Metrics Across All Folds:")
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")



📦 Fold 1/5 ------------------------


[Fold 1] Epoch 1/2: 100%|██████████| 2329/2329 [07:12<00:00,  5.38it/s]


✅ Epoch 1 | Train Acc: 0.8709
💾 Model saved to: saved_models/fold1_epoch1.pth
🧪 Validation Acc (Fold 1, Epoch 1): 0.9259


[Fold 1] Epoch 2/2: 100%|██████████| 2329/2329 [07:10<00:00,  5.41it/s]


✅ Epoch 2 | Train Acc: 0.9076
💾 Model saved to: saved_models/fold1_epoch2.pth
🧪 Validation Acc (Fold 1, Epoch 2): 0.9123

📦 Fold 2/5 ------------------------


...

✅ Epoch 2 | Train Acc: 0.9074
💾 Model saved to: saved_models/fold4_epoch2.pth
🧪 Validation Acc (Fold 4, Epoch 2): 0.9240

📦 Fold 5/5 ------------------------


[Fold 5] Epoch 1/2: 100%|██████████| 2329/2329 [07:11<00:00,  5.40it/s]


✅ Epoch 1 | Train Acc: 0.9016
💾 Model saved to: saved_models/fold5_epoch1.pth
🧪 Validation Acc (Fold 5, Epoch 1): 0.9189


[Fold 5] Epoch 2/2: 100%|██████████| 2329/2329 [07:10<00:00,  5.41it/s]


✅ Epoch 2 | Train Acc: 0.9148
💾 Model saved to: saved_models/fold5_epoch2.pth
🧪 Validation Acc (Fold 5, Epoch 2): 0.8777

📊 Final Evaluation Metrics Across All Folds:
Accuracy: 0.9066
Precision: 0.9051
Recall: 0.9066
F1 Score: 0.9030


In [ ]:
import matplotlib.pyplot as plt

epochs_range = list(range(1, len(fold_train_losses) + 1))

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, fold_train_losses, marker='o', label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, fold_train_accuracies, marker='o', label='Train Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
torch.save(model.state_dict(), "model_weights.pth")
